In [0]:
import pandas as pd                                                                 # Import pandas for data cleaning
import numpy as np                                                                  # Import Numpy for Maths functions
import matplotlib.pyplot as plt                                                     # Import Matplot for Viz functions
import seaborn as sns                                                               # Import Seaborn for visualization
import matplotlib.pyplot as plt                                                     # Import matplotlib library for visualization
import plotly.express as px                                                         # Import plotly library for visualization
import plotly.graph_objects as go

from sklearn.ensemble import IsolationForest                                        # EDA-Isolation Forest Analysis Functions

from pyspark.sql import functions as F
from pyspark.sql.functions import col, StringType, NumericType                      # TableFunctions
from pyspark.sql.functions import mean, min, max, stddev, count, sum as _sum        # MathsFunctions
from pyspark.sql.functions import to_date, year, month, datediff                    # DateFunctions
from pyspark.sql.functions import abs                                               # OtherFunctions

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

from sklearn.linear_model import LinearRegression                                   # LinearRegression Analysis Functions
from sklearn.metrics import r2_score, mean_squared_error                            # LinearRegression Analysis Functions

from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder        # Classification Analysis Functions
from pyspark.ml import Pipeline                                                     # Classification Analysis Functions
from sklearn.linear_model import LogisticRegression                                 # Classification Analysis Functions
from sklearn.metrics import mean_squared_error, r2_score                            # Classification Analysis Functions
from sklearn.impute import SimpleImputer                                            # Classification Analysis Functions
from sklearn.preprocessing import LabelEncoder                                      # Classification Analysis Functions
from sklearn.preprocessing import OneHotEncoder                                     # Classification Analysis Functions
from sklearn.preprocessing import StandardScaler                                    # Classification Analysis Functions
from sklearn.model_selection import train_test_split                                # Classification Analysis Functions
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report # Classification Analysis Functions
from sklearn.metrics import roc_curve, auc                                          # Classification Analysis Functions

from sklearn.tree import DecisionTreeClassifier                                     # DecisionTree Analysis Functions

from sklearn.ensemble import RandomForestClassifier                                 # RandomForest Analysis Functions

from sklearn.cluster import KMeans                                                  # KMeans Cluster Analysis Functions


In [0]:
df = spark.read.table("titanicdata.titanic_bronze.titanic")
df_raw = df
# display(df)

In [0]:
# Encode gender safely
df = df.withColumn("gender", F.when(col("Sex")=="male",1).otherwise(0))
df = df.drop("Sex")

# De-duplication
df = df.dropDuplicates()

# Separate categorical and numeric columns
categorical_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]

# Fill categorical with mode
for col_name in categorical_cols:
    mode_row = df.groupBy(col_name).count().orderBy(F.desc("count")).first()
    if mode_row and mode_row[0] is not None:
        df = df.fillna({col_name: mode_row[0]})

# Fill numeric with median
for col_name in numeric_cols:
    median_val = df.approxQuantile(col_name, [0.5], 0.01)[0]
    if median_val is not None:
        df = df.fillna({col_name: median_val})

# Save all columns to Silver table
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "titanicdata.titanic_silver.titanicdata"
)
# display(df)

In [0]:
#HANDLING NULLS
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
])
null_counts.write.mode("overwrite").saveAsTable("titanicdata.titanic_silver.titanicdata_nulls")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# Numeric columns to check
numeric_cols = [c for (c, t) in df.dtypes if t in ("int", "double", "float")]

# Function to add outlier flag for a column
def add_outlier_flag(df, col):
    q1, q3 = df.approxQuantile(col, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    return df.withColumn(
        f"{col}_Outlier",
        F.when((F.col(col) < lower_bound) | (F.col(col) > upper_bound), 1).otherwise(0)
    )

# Apply outlier flagging for each numeric column
for col_name in numeric_cols:
    df = add_outlier_flag(df, col_name)

# Show sample with flags
df.select(numeric_cols + [f"{c}_Outlier" for c in numeric_cols]).show(10)

# Avoid duplicate PassengerId
selected_cols = [c for c in df.columns[12:] if c != "PassengerId"]
titanicdata_outliers = df.select("PassengerId", *selected_cols)

# Force overwrite with schema alignment
titanicdata_outliers.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "titanicdata.titanic_silver.titanicdata_outliers")

In [0]:
# Example Titanic dataset
# Select numeric features
pdf = df_raw.select("Age", "Fare").toPandas()

# Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
outlier_pred = iso.fit_predict(pdf)

# Add outlier flag to Spark DataFrame
outlier_pred_list = outlier_pred.tolist()
df_with_id = df_raw.withColumn("row_id", F.monotonically_increasing_id())
outlier_pred_df = spark.createDataFrame(
    [(i, int(flag)) for i, flag in enumerate(outlier_pred_list)],
    ["row_id", "IForest_Outlier"]
)
df = df_with_id.join(outlier_pred_df, on="row_id", how="left").drop("row_id")

# Save to Silver table
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "titanicdata.titanic_silver.titanicdata_anomolies"
)

#display(df)